In [ ]:
# ==============================================================================
# OFFICE FOR NATIONAL STATISTICS (ONS) - CENSUS 2021
# 
# ARCHITECTURE NOTE: ONS publishes the "Religion (detailed)" census variables 
# as flat HTML tables on a public webpage. There is no formal LOD or API endpoint.
# The page contains two distinct classifications: 
# 1. Religion (191 categories)
# 2. Religion 58a (58 categories)
#
# SSSOM ALIGNMENT STRATEGY: 
# This script uses a modified Strategy B (Web Scraping). It fetches the HTML 
# and parses the tables directly into pandas DataFrames. 
# 
# To handle overlapping codes between the two tables (and satisfy SSSOM identifier 
# rules without hallucinating data), the script prepends the classification's 
# mnemonic to the Concept_ID (e.g., "religion-001" and "religion_58a-34").
# 
# Strict Extraction Rule: Labels containing colons (e.g., "Other religion: Pagan") 
# are extracted exactly as they appear in the source, rather than synthesized 
# into artificial parent-child hierarchies.
#
# Operational Code Filtering: Survey-administrative codings (e.g., negative codes 
# for errors, "Not answered") are intentionally cleaned and excluded. As a result, 
# the final extracted row counts will be slightly lower than the official 191 
# and 58 category totals.
# ==============================================================================

import os
import sys
import pandas as pd
import requests
import time
from dotenv import load_dotenv
from io import StringIO

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ONS"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="Office for National Statistics",
    fallback_uri="" # Intentional blank for non-LOD sources
)
SOURCE_NAME = registry_data['Source_Name']

raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
ONS_URL = "https://www.ons.gov.uk/census/census2021dictionary/variablesbytopic/ethnicgroupnationalidentitylanguageandreligionvariablescensus2021/religiondetailed/classifications"

HEADERS = {
    'User-Agent': 'ReligiousMappingProject/1.0',
    'Accept': 'text/html'
}

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
    else:
        print(f"[*] Found existing ONS file. Overwriting for fresh scrape...")
        os.remove(raw_ingest_file)

# --- 4. Extraction & Parsing ---
print(f"[*] Fetching ONS classification tables from {ONS_URL}...")

try:
    response = requests.get(ONS_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    # Read HTML tables. Converters ensure '001' doesn't become integer 1.
    html_io = StringIO(response.text)
    tables = pd.read_html(html_io, converters={'Code': str})
    print(f"[*] Successfully located {len(tables)} tables on the page.")
except Exception as e:
    print(f"[!] CRITICAL ERROR: Failed to retrieve or parse HTML tables. Exception: {e}")
    sys.exit(1)
  
# The ONS page defines two specific tables
# Table 0: Religion (detailed) classification (Mnemonic: religion)
# Table 1: Religion (detailed) classification 58a (Mnemonic: religion_58a)
table_mnemonics = ["religion", "religion_58a"]

all_rows = []

for idx, df in enumerate(tables):
    if idx >= len(table_mnemonics):
        break # Ignore any extra stray tables on the page
        
    mnemonic = table_mnemonics[idx]
    print(f"[*] Processing table '{mnemonic}' ({len(df)} rows)...")
    
    for _, row in df.iterrows():
        code = str(row.get('Code', '')).strip()
        label = str(row.get('Name', '')).strip()
        
        # --- Clean Operational Codes ---
        # Skip negative codes (-1, -6, -9, etc.), "Not answered" (900/57), and empty rows
        if not code or code.startswith('-') or "Not answered" in label:
            continue
            
        # Prepend mnemonic to code to ensure absolute uniqueness across tables
        concept_id = f"{mnemonic}-{code}"

        extracted_data = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': label,
            'CURIE': f"{SOURCE_PREFIX}:{concept_id}",
            'Formal_Label': "", # Left blank: ONS provides no distinct FSN
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': label, # Strict Extraction Rule: No synthetic hierarchy
            'Synonyms': "",
            'Description': "",
            'Parent_IDs': "", # Flat catalog
            'Concept_ID': concept_id,
            'URI': "", # Intentional Blank: ONS is not LOD
            'Has_Translation': "",
            'Status': "active",
            'Crosswalks': ""
        }
        
        all_rows.append(finalize_row(extracted_data))

# --- 5. Export to Bronze Layer ---
if all_rows:
    export_df = pd.DataFrame(all_rows)[COLUMN_ORDER]
    export_df.to_csv(raw_ingest_file, index=False, encoding='utf-8-sig')
    print(f"\n[COMPLETE] Successfully wrote {len(all_rows)} valid concepts to {raw_ingest_file}")
else:
    print("\n[!] CRITICAL ERROR: No valid data rows were parsed from the ONS tables.")